<a href="https://colab.research.google.com/github/amosagekouassi-source/DI-Bootcamp/blob/master/Custom_Attention_SMS_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: Custom Attention Mechanism & SMS Spam Classification

Welcome to the guided notebook for the *Custom Attention Mechanism & SMS Spam* daily challenge. Cells tagged as **PREFILLED** are ready to run as-is. Cells tagged as **To-Do** require you to replace the placeholder code or text with your own work before executing the notebook.


## Why are we doing this?
Modern NLP systems rely on attention. By rolling your own attention block and contrasting it with a pre-trained GPT-2 classifier, you will demystify how query/key/value flows shape downstream predictions on a real SMS spam dataset.

![Image](https://github.com/user-attachments/assets/bc4d5315-983b-4fc1-9011-25fa743bb25f)


## Learning objectives
- Implement a custom scaled dot-product attention layer from scratch.
- Explain the respective roles of queries, keys, and values.
- Fine-tune GPT-2 for binary spam classification and compare it to a custom model.
- Evaluate both systems with accuracy, precision, recall, and F1.
- Reflect on trade-offs between transformer-based and lightweight attention models.


> **Learning point**
> Work through each part sequentially. Replace every `# TODO:` marker before running the cell so that downstream steps (tokenization, training, evaluation) receive the expected inputs.


# Part 1: Setup & Data Loading
As on the platform, start by installing dependencies, importing helper modules, and slicing the SMS dataset into 4,000 training rows and 1,000 validation rows.


**PREFILLED: run once**
Installs the libraries required for this challenge.


In [3]:
%pip install --quiet datasets evaluate transformers[sentencepiece] accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00


**To-Do (code)**
Import pandas plus the dataset utilities exactly as in the platform instructions.


In [4]:
import pandas as pd
from datasets import Dataset

**To-Do (code)**
Load the UCI SMS Spam parquet file, convert it to a Hugging Face Dataset, then build 4,000 / 1,000 splits as described in the enoncé.


In [5]:
DATA_PATH = 'hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet'
df = pd.read_parquet(DATA_PATH)
hf_dataset = Dataset.from_pandas(df)

TRAIN_START = 0
TRAIN_END = 4000
VAL_START = 4000
VAL_END = 5000

train_ds = hf_dataset.select(range(TRAIN_START, TRAIN_END))
val_ds = hf_dataset.select(range(VAL_START, VAL_END))
display(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,sms,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...\n,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


# Part 2: Tokenization Setup
Initialize the GPT-2 tokenizer, set a padding token, and prepare batched tokenization for both splits.


> **Learning point**
> GPT-2 does not define a pad token. Reusing the EOS token keeps inputs aligned with how the model was pretrained.


In [6]:
from transformers import GPT2Tokenizer

MODEL_NAME = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:
TEXT_COLUMN = 'sms'
PADDING_STRATEGY = 'max_length'
TRUNCATION_FLAG = True
MAX_SEQ_LEN = 64

def tokenize_fn(examples):
    return tokenizer(
        examples[TEXT_COLUMN],
        padding=PADDING_STRATEGY,
        truncation=TRUNCATION_FLAG,
        max_length=MAX_SEQ_LEN,
    )

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Part 3: Pre-trained GPT-2 Classifier
Load GPT-2 with a classification head suited for binary spam detection.


In [8]:
import torch
from transformers import GPT2ForSequenceClassification

NUM_LABELS = 2
model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    pad_token_id=tokenizer.eos_token_id,
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Part 4: Custom Attention Implementation
Build the simple attention layer, classifier, and data pipeline for the scratch model.


> **Learning point**
> Scaling the dot products by $1/\sqrt{d_k}$ keeps gradients stable and prevents the softmax from collapsing when embeddings grow. This opeeration is crucial for training deep attention models.

In [9]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, value), attn

class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.attn = Attention(embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embed = self.embedding(x)
        attn_output, _ = self.attn(embed, embed, embed)
        pooled = attn_output.mean(dim=1)
        return self.fc(pooled)

> **Learning point**
> Tokenize once and reuse the same 64-token cap so both models receive comparable context windows.


In [10]:
ATTN_TEXT_COLUMN = 'sms'
ATTN_MAX_LEN = 64

def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example[ATTN_TEXT_COLUMN],
        max_length=ATTN_MAX_LEN,
        truncation=True,
        padding='max_length',
    )
    return {'input_ids': tokens, 'label': example['label']}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn = val_ds.map(preprocess_for_attention)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [11]:
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label': torch.tensor(item['label'], dtype=torch.long),
        }

TRAIN_DATA_FOR_LOADER = train_ds_attn
VAL_DATA_FOR_LOADER = val_ds_attn

train_loader = DataLoader(SMSDataset(TRAIN_DATA_FOR_LOADER), batch_size=32, shuffle=True)
val_loader = DataLoader(SMSDataset(VAL_DATA_FOR_LOADER), batch_size=32)

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import GPT2Tokenizer

# Re-initialize tokenizer if it was lost from memory
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab_size = len(tokenizer)
embed_dim = 64
num_classes = 2
learning_rate = 1e-3

attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)
optimizer = optim.Adam(attn_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Training Loop
attn_model.train()
for batch in train_loader:
    inputs = batch["input_ids"].to(device)
    labels = batch["label"].to(device)
    optimizer.zero_grad()
    outputs = attn_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print(f"Custom Attention model trained successfully. Final batch loss: {loss.item():.4f}")

Custom Attention model trained successfully. Final batch loss: 0.1827


# Part 5: Metrics & Evaluation
Load accuracy, precision, recall, and F1 from `evaluate`, then implement the shared `compute_metrics` helper.


In [13]:
import evaluate
import numpy as np

accuracy = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall = evaluate.load('recall')
f1 = evaluate.load('f1')

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall': recall.compute(predictions=preds, references=labels)['recall'],
        'f1': f1.compute(predictions=preds, references=labels)['f1'],
    }

> **Learning point**
> Use the same helper dictionary pattern for both GPT-2 and the custom model so you can compare metrics side by side.


In [18]:
from torch.utils.data import DataLoader
from datasets import config
import torch

# Désactive la détection de torchvision pour éviter l'erreur VideoReader
config.TORCHVISION_AVAILABLE = False

# S'assure que les dépendances sont disponibles
if 'val_tok' not in globals():
    if 'val_ds' in globals() and 'tokenize_fn' in globals():
        val_tok = val_ds.map(tokenize_fn, batched=True)
    else:
        raise NameError("val_ds ou tokenize_fn introuvable.")

val_loader_gpt2 = DataLoader(val_tok.with_format("torch"), batch_size=32)

gpt2_preds = []
gpt2_labels = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

for batch in val_loader_gpt2:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    mask = batch['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(inputs, attention_mask=mask)
        logits = outputs.logits

    preds = torch.argmax(logits, dim=-1)
    gpt2_preds.extend(preds.cpu().tolist())
    gpt2_labels.extend(labels.cpu().tolist())

gpt2_metrics = {
    'accuracy': accuracy.compute(predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall': recall.compute(predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1': f1.compute(predictions=gpt2_preds, references=gpt2_labels)['f1'],
}
print('GPT-2 Metrics:', gpt2_metrics)

GPT-2 Metrics: {'accuracy': 0.861, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
attn_preds = []
attn_labels = []

# Ensure model is in eval mode and defined
if "attn_model" in globals():
    attn_model.eval()
    for batch in val_loader:
        inputs = batch["input_ids"].to(device)
        labels = batch["label"].to(device)
        with torch.no_grad():
            outputs = attn_model(inputs)
            preds = torch.argmax(outputs, dim=1)
        attn_preds.extend(preds.cpu().tolist())
        attn_labels.extend(labels.cpu().tolist())

    attn_metrics = {
        "accuracy": accuracy.compute(predictions=attn_preds, references=attn_labels)["accuracy"],
        "precision": precision.compute(predictions=attn_preds, references=attn_labels)["precision"],
        "recall": recall.compute(predictions=attn_preds, references=attn_labels)["recall"],
        "f1": f1.compute(predictions=attn_preds, references=attn_labels)["f1"],
    }
    print("Attention Model Metrics:", attn_metrics)
else:
    print("Error: attn_model not found. Please run the training cell first.")

Attention Model Metrics: {'accuracy': 0.859, 'precision': 0.42857142857142855, 'recall': 0.04316546762589928, 'f1': 0.0784313725490196}


# Part 6: Reflection Questions
Answer directly in the markdown cells below once your experiments finish.


### 1. What are the roles of query, key, and value in the attention mechanism?
- **Query (Q)**: Represents the current token looking for relevant information.
- **Key (K)**: Represents all tokens in the sequence, acting as a label to match against the query.
- **Value (V)**: Contains the actual information that will be weighted and summed based on the match between Q and K.

### 2. Why do we use a scaling factor in the dot-product attention?
We use $1/\sqrt{d_k}$ to prevent the dot products from growing too large in magnitude as the embedding dimension increases. Without it, the softmax function would enter regions with very small gradients, leading to the 'vanishing gradient' problem during training.

### 3. How does self-attention differ from traditional sequence models like RNNs?
Unlike RNNs, which process tokens sequentially (O(n) time for long-range dependencies), self-attention processes all tokens in parallel. This allows for constant-time path lengths between any two tokens, making it much more efficient at capturing long-range dependencies and better suited for GPU acceleration.

### 4. Performance analysis
GPT-2 generally performs better due to its pre-trained knowledge and deeper architecture. The custom model is lightweight and faster but lacks context. An improvement for the custom model would be to add multi-head attention and position embeddings to help it understand the order of words in the SMS.